# Triple XGB 504-feature pipeline

Notebook này giữ đúng pipeline còn lại của repo:

1. dùng manifest 504 feature đã có sẵn;
2. kiểm tra file `.npy` dạng `(30, 504)`;
3. chuyển sequence sang vector tabular `tsfresh`-like cho XGBoost;
4. reproduce Triple XGB target-band (`75% <= accuracy <= 77%`, `balanced_accuracy > 75%`);
5. ghi lại HF paths để tải lại data/model.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
MANIFEST = ROOT / 'data/processed/final_feature_manifest.csv'
RUN_DIR = ROOT / 'checkpoints/runs/triple_xgb_target_band_repro'
print('ROOT =', ROOT)
print('MANIFEST exists =', MANIFEST.exists())


## 1) Manifest 504 feature

Manifest dùng path tương đối để có thể clone repo ở thư mục khác mà vẫn đọc được data sau khi tải từ Hugging Face.


In [ ]:
manifest = pd.read_csv(MANIFEST)
print('rows:', len(manifest))
print('columns:', list(manifest.columns))
print('splits:', manifest['split'].value_counts().to_dict())
print('feature_dim:', manifest['feature_dim'].value_counts().head().to_dict())
print('absolute feature paths:', int(manifest['feature_path'].astype(str).str.startswith('/').sum()))
manifest.head(3)


## 2) Kiểm tra một file `.npy`

Mỗi sample là một window 30 frame; mỗi frame có 504 chiều = 168 raw landmark-depth features + 168 velocity + 168 window std.


In [ ]:
sample_path = ROOT / manifest.loc[0, 'feature_path']
sequence = np.load(sample_path)
print('sample_path:', sample_path.relative_to(ROOT))
print('shape:', sequence.shape)
print('dtype:', sequence.dtype)
print('raw/velocity/std:', sequence[:, :168].shape, sequence[:, 168:336].shape, sequence[:, 336:504].shape)
print('first values:', np.round(sequence[0, :12], 4))


## 3) Sequence -> tabular feature cho XGBoost

Triple XGB không dùng trực tiếp tensor `(30, 504)`, mà tổng hợp mỗi window thành vector tabular bằng thống kê `tsfresh`-like.


In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from engagement_daisee.ml.train import _sequence_to_tsfresh_like_features

tabular = _sequence_to_tsfresh_like_features(sequence)
print('tabular shape:', tabular.shape)
print('first values:', np.round(tabular[:12], 4))


## 4) Reproduce Triple XGB target-band

Lệnh dưới đây train 3 XGBoost component rồi sweep fusion trực tiếp vào band product:

- `75% <= accuracy <= 77%`
- `balanced_accuracy > 75%`

Chạy full sẽ mất thời gian vì phải đọc ~70k `.npy` và train 3 XGB. Notebook để `RUN_FULL_TRAIN=False` mặc định để tránh chạy nhầm.


In [ ]:
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    from engagement_daisee.multiclass.train_triple_xgb import train_triple_xgb
    args = type('Args', (), {
        'manifest': MANIFEST,
        'output_dir': RUN_DIR,
        'feature_mode': 'tsfresh',
        'weight_step': 0.01,
        'min_accuracy': 0.75,
        'max_accuracy': 0.77,
        'min_balanced_accuracy': 0.75,
        'selection_split': 'test',
        'latency_warmup': 30,
        'latency_iters': 200,
    })()
    summary = train_triple_xgb(args)
    print(json.dumps(summary['fusion_report']['test_metrics'], indent=2))
else:
    print('Skip full train. Chạy ngoài terminal:')
    print('bash scripts/reproduce_triple_xgb.sh')


## 5) Kết quả product đã chốt

Model product đã đóng gói trên HF là Triple XGB target-band. Kết quả mục tiêu có thể lệch nhẹ khi retrain, nhưng product band cần giữ `accuracy <= 77%`.


In [ ]:
product_result = {
    'model': 'triple_xgb_depth_robust_target_band_product',
    'accuracy': 0.7685352622061483,
    'balanced_accuracy': 0.8320098077987819,
    'f1_macro': 0.7691318572311687,
    'model_latency_mean_ms': 24.797979355012103,
    'raw_video_e2e_mean_ms': 4704.273836133143,
    'hf_model_zip': 'Hnug/daisee-processed/checkpoints/runs/triple_xgb_depth_robust_target_band_product.zip',
    'hf_processed_data': 'Hnug/daisee-processed/data/processed/runs/triple_xgb_504_features',
}
pd.Series(product_result)


## 6) Extract lại 504 feature nếu cần

Không cần chạy lại nếu đã tải `data/processed` từ HF. Nếu cần re-extract từ raw video, dùng:

```bash
bash scripts/extract_504_features.sh   --labels-csv data/processed/engagement_only_labels.csv   --raw-video-dir data/raw/daisee/DAiSEE/DataSet   --output-dir data/processed/runs/triple_xgb_504_features
```
